# Phase 6A | Conference-Focused Controller Failure Benchmark

**Aim:** Compare tuned and development-dose-matched threshold control against nominal and gain-uncertainty-aware MPC under incorrect stimulation-response assumptions. **No model retraining.**

**Evidence:** Spontaneous rate samples from DANDI 000568 (two sessions, **one mouse**) anchor the simulation. All intervention effects are **assumed / simulated**, not learned or experimentally validated. The controller uses a train-session-fitted AR(1) surrogate rather than the frozen GRU in the feedback loop. No epilepsy or patient-specific claims.

**Runs without internet or extra Kaggle inputs.** CPU is sufficient. Full mode uses 16 recorded starting histories per direction and 6 numerical noise seeds; quick mode uses fewer only for debugging.

**Execution rule:** This notebook is configured to run in Kaggle only. No experiment outputs are included. Run the full notebook in Kaggle and use the exported ZIP as the sole results source.


In [1]:
# Settings (CPU; zero extra inputs needed)
from pathlib import Path
import os,base64,hashlib
QUICK_MODE = False  # True: code smoke test only. False: final, reproducible computational benchmark.
os.environ['PHASE6A_QUICK']='1' if QUICK_MODE else '0'
ROOT=Path('/kaggle/working')
if not ROOT.is_dir():
    raise RuntimeError('Phase 6A experiments must run in Kaggle. Upload this notebook to Kaggle and use Run All.')
os.environ['PHASE6A_OUT']=str(ROOT/'phase6a_conference_outputs')
bundle_path=ROOT/'Phase6A_Frozen_Real_Evidence.zip'
BUNDLE_DATA_B64='...'
bundle_bytes=base64.b64decode(BUNDLE_DATA_B64)
assert hashlib.sha256(bundle_bytes).hexdigest()=='2e4d294ce0c9e8f01d20639c63738ce217273eff8cb75a763446f7ee61ff0be1', 'Embedded evidence checksum failed'
bundle_path.write_bytes(bundle_bytes)
os.environ['PHASE6A_BUNDLE']=str(bundle_path)
print('QUICK_MODE =',QUICK_MODE,' evidence bundle size',round(bundle_path.stat().st_size/1024,1),'KiB')


QUICK_MODE = False  evidence bundle size 106.7 KiB


## Conference benchmark: uncertainty-aware controllers under sign/gain mismatch

In [2]:
"""Phase 6A, simulated interventions anchored to observed (no-light) mouse firing-rate samples.
CAUTION: Not biological optogenetic control. Dimensionaless action; optical gains are assumed.
"""
import os, io, json, zipfile, hashlib, math, itertools, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT=Path(os.environ.get('PHASE6A_OUT','/kaggle/working/phase6a_conference_outputs' if Path('/kaggle/working').exists() else '/mnt/data/Phase6A_Conference/results'))
OUT.mkdir(parents=True,exist_ok=True)
QUICK_MODE=os.environ.get('PHASE6A_QUICK','0')=='1'
# Full mode expands numerical repeats, not the number of independent animals.
N_TEST_ANCHORS=8 if QUICK_MODE else 16
N_DEV_ANCHORS=5 if QUICK_MODE else 10
N_SEEDS=2 if QUICK_MODE else 6
N_DEV_SEEDS=1 if QUICK_MODE else 2
BOOT_REPS=200 if QUICK_MODE else 700
SEED=413271
DT_SEC=1.0
STEPS=30
HISTORY=8
ACTION_VALUES=(0.,.5,1.)
BUDGET=12.
REFRACTORY=2
HORIZON=3
LAMBDA_ACTION=.055
LAMBDA_CHANGE=.004
POLICIES=('NO_LIGHT','FIXED','THRESHOLD_TUNED','THRESHOLD_DOSE_MATCHED','MPC_NOMINAL','MPC_RISK_STATIC','MPC_RISK_ADAPTIVE')
PLANTS=('AR1','REPLAY')
SCENARIOS=('STRONG_INHIBITION','WEAK_INHIBITION','ZERO_EFFECT','EXCITATION','SIGN_REVERSAL','ACTUATOR_SLOW')
BUNDLE=Path(os.environ.get('PHASE6A_BUNDLE', '/mnt/data/Phase6A_Conference/Phase6A_Frozen_Real_Evidence.zip'))
if not BUNDLE.exists():
    BUNDLE=Path('/kaggle/working/Phase6A_Frozen_Real_Evidence.zip')
assert BUNDLE.exists(), f'Missing frozen real-data evidence bundle: {BUNDLE}'
with zipfile.ZipFile(BUNDLE) as z:
    observed=pd.read_csv(io.BytesIO(z.read('OBSERVED_spontaneous_rate_snapshots_1s.csv')))
    trainpars=pd.read_csv(io.BytesIO(z.read('REAL_train_derived_baseline_parameters.csv')))
    provenance=json.loads(z.read('PROVENANCE.json'))
assert len(observed)>1000 and set(observed.fold)=={1,2} and provenance['independent_animals']==1

## Train/test separation, empirical anchor sequences

In [3]:
def contiguous_episodes(data, n, rng, nsteps=STEPS, nhist=HISTORY):
    d=data[['t_sec','observed_Hz']].dropna().drop_duplicates('t_sec').sort_values('t_sec').reset_index(drop=True)
    t=d.t_sec.to_numpy(dtype=float);y=d.observed_Hz.to_numpy(dtype=float)
    if not len(y):raise RuntimeError('No empirical observations')
    groups=[];beg=0
    for i in range(1,len(y)):
        if abs((t[i]-t[i-1])-1.)>1e-5:
            if i-beg>=nhist+nsteps: groups.append((beg,i))
            beg=i
    if len(y)-beg>=nhist+nsteps:groups.append((beg,len(y)))
    candidates=[]
    for a,b in groups:
        candidates.extend(range(a+nhist,b-nsteps+1,6))
    rng.shuffle(candidates)
    starts=[]
    for anchor in candidates:
        if any(abs(anchor-k)<(nsteps+nhist+5) for k in starts):continue
        starts.append(anchor)
        if len(starts)>=n:break
    if len(starts)<n:raise RuntimeError(f'Only {len(starts)} of {n} independent-in-time anchors available')
    out=[]
    for i,a in enumerate(sorted(starts)):
        hist=np.array(y[a-nhist:a],dtype=float)
        replay=np.array(y[a:a+nsteps],dtype=float)
        assert len(hist)==nhist and len(replay)==nsteps
        out.append({'anchor_id':i,'start_t_sec':float(t[a]),'history':hist,'replay':replay,'first_sample_rate':float(hist[-1])})
    return out

def fit_training_parameters(data, baseline_par):
    d=data.sort_values('t_sec');t=d.t_sec.to_numpy(float);y=d.observed_Hz.to_numpy(float)
    m=np.abs(np.diff(t)-1.)<1e-5
    a=y[:-1][m];b=y[1:][m]
    mu=float(baseline_par.mu_Hz)
    denom=float(np.sum((a-mu)**2))
    phi=float(np.clip(np.sum((a-mu)*(b-mu))/max(denom,1e-6),.12,.82))
    resid=b-(mu+phi*(a-mu))
    noise=float(np.clip(np.std(np.clip(resid,*np.quantile(resid,[.02,.98]))),.2,2.5))
    scale=float(max(.35,np.std(np.clip(y,*np.quantile(y,[.01,.99])))))
    target=float(np.quantile(y,.45))
    return {'mu':mu,'phi':phi,'noise':noise,'scale':scale,'target':target,
            'source_session':str(data.heldout_session.iloc[0]),
            'n_training_snapshot_rows':int(len(y))}

FOLDS={}
for fold in (1,2):
    test=observed[observed.fold==fold].copy()
    train=observed[observed.fold==3-fold].copy()
    assert test.heldout_session.iloc[0]!=train.heldout_session.iloc[0]
    pars=fit_training_parameters(train,trainpars[trainpars.fold==fold].iloc[0])
    FOLDS[fold]={'pars':pars,'train_session':str(train.heldout_session.iloc[0]),'test_session':str(test.heldout_session.iloc[0]),
                 'dev':contiguous_episodes(train,N_DEV_ANCHORS,np.random.default_rng(SEED+fold)),
                 'test':contiguous_episodes(test,N_TEST_ANCHORS,np.random.default_rng(SEED+fold+100))}
    print('Fold',fold,'TRAIN',FOLDS[fold]['train_session'],'TEST',FOLDS[fold]['test_session'],
          'mu',round(pars['mu'],3),'target',round(pars['target'],3),'noise',round(pars['noise'],3),
          'dev/test anchors',len(FOLDS[fold]['dev']),len(FOLDS[fold]['test']))
assert all(p['train_session']!=p['test_session'] for p in FOLDS.values())

Fold 1 TRAIN session_9 TEST session_12 mu 1.87 target 1.579 noise 1.094 dev/test anchors 10 16
Fold 2 TRAIN session_12 TEST session_9 mu 2.051 target 1.915 noise 1.394 dev/test anchors 10 16


## Predeclared synthetic plants and optogenetic gain faults

In [4]:
def true_gain(scenario, step, pars):
    scale=pars['scale']
    if scenario=='STRONG_INHIBITION':return -1.0*scale
    if scenario=='WEAK_INHIBITION':return -.28*scale
    if scenario=='ZERO_EFFECT':return 0.
    if scenario=='EXCITATION':return +.38*scale
    if scenario=='SIGN_REVERSAL':return (-.80 if step<STEPS//3 else +.45)*scale
    if scenario=='ACTUATOR_SLOW':return -.75*scale
    raise KeyError(scenario)

def make_disturbances(fold,aid,seed,plant,pars):
    # Same noise/shock for all scenarios and policies with the same tuple.
    rng=np.random.default_rng(SEED+100000*fold+1000*aid+17*seed+(0 if plant=='AR1' else 7))
    eps=rng.normal(0,pars['noise']*.6,STEPS)
    shock=np.zeros(STEPS)
    shock[STEPS//3:STEPS//3+4]=1.3*pars['scale']
    shock[2*STEPS//3:2*STEPS//3+3]=.7*pars['scale']
    return eps,shock

## Frozen, causal controller rules (no held-out future neural activity used)

In [5]:
PLANS=list(itertools.product(ACTION_VALUES,repeat=HORIZON))

def nominal_forecast(y,pars):
    return max(0.,pars['mu']+pars['phi']*(y-pars['mu']))

def mpc_action(history,z,remaining,last_on,last_action,t,pars,gain_assumed):
    """Receding-horizon 3-step exhaustive search; only first action executed."""
    target=pars['target']; scale=pars['scale']
    best=np.inf; first=0.
    for plan in PLANS:
        used=0.;laston=last_on;valid=True;y=float(history[-1]); zz=float(z);last=last_action;total=0.
        for k,a in enumerate(plan):
            if a>0:
                if used+a>remaining+1e-8 or (t+k-laston)<REFRACTORY:
                    valid=False;break
                laston=t+k
            used+=a
            zz=.35*zz+.65*a
            y=nominal_forecast(y,pars)+gain_assumed*zz
            y=float(max(0,y))
            total+=(max(0.,y-target)/scale)**2+LAMBDA_ACTION*a+LAMBDA_CHANGE*(a-last)**2
            last=a
        if not valid:continue
        if total<best-1e-12 or (abs(total-best)<1e-12 and plan[0]<first):
            best=total;first=float(plan[0])
    return first

def create_policy_state(pars):
    # Prior confidence intentionally wrong in some test scenarios; this is a failure-mode experiment.
    g0=-.80*pars['scale'];sd=.30*pars['scale']
    return {'mu_g':g0,'var_g':sd**2,'prior_mu':g0,'prior_var':sd**2,
            'z':0.,'last_action':0.,'last_on':-999,'used':0.,'sign_abstains':0,'risk_ci_log':[]}

def bayes_update(state,pars,y_prev,y_new,z):
    # Posterior only observes controlled plant data; it never sees true g or replay future.
    if abs(z)<.075:return
    prediction=nominal_forecast(y_prev,pars)
    residual=float(np.clip(y_new-prediction,-3.*pars['noise'],3.*pars['noise']))
    # Imperfect natural-dynamics predictor is an explicitly simulated confound.
    meas_var=(1.5*pars['noise'])**2
    oldp=1./max(1e-8,state['var_g'])
    newp=oldp+(z*z)/meas_var
    mean=(state['mu_g']*oldp+z*residual/meas_var)/newp
    state['mu_g']=float(mean);state['var_g']=float(1./newp)

def select_action(policy,hist,t,state,pars,threshold_cfg):
    left=BUDGET-state['used']
    if left<.5-1e-9 or t-state['last_on']<REFRACTORY:return 0.,False
    if policy=='NO_LIGHT':return 0.,False
    if policy=='FIXED':return (.5 if t%6==0 else 0.),False
    if policy in ('THRESHOLD_TUNED','THRESHOLD_DOSE_MATCHED'):
        cfg=threshold_cfg['tuned' if policy=='THRESHOLD_TUNED' else 'dose_matched']
        u=cfg['amplitude'] if np.mean(hist[-3:])>cfg['threshold'] else 0.
        return min(u,left),False
    if policy=='MPC_NOMINAL':
        return mpc_action(hist,state['z'],left,state['last_on'],state['last_action'],t,pars,-.8*pars['scale']),False
    if policy.startswith('MPC_RISK_'):
        mg=state['mu_g'];sd=math.sqrt(max(0,state['var_g']))
        upper=mg+1.28*sd
        # If sign could be non-inhibitory, abstain instead of assuming safe stimulation.
        if upper>=-.10*pars['scale']:
            return 0.,True
        return mpc_action(hist,state['z'],left,state['last_on'],state['last_action'],t,pars,upper),False
    raise KeyError(policy)

## Paired simulated episodes; policies never see future baseline or ground-truth gain

In [6]:
def simulate_episode(anchor,fold,seed,plant,scenario,policy,pars,threshold_cfg,with_trace=False):
    ep,sh=make_disturbances(fold,anchor['anchor_id'],seed,plant,pars)
    hist=list(anchor['history'].copy())
    replay=anchor['replay']
    prev_replay=float(hist[-1])
    state=create_policy_state(pars)
    values=[];actions=[];forbidden_actions=[];abstentions=[];time_bad=[];trace=[]
    for t in range(STEPS):
        u,abstain=select_action(policy,hist,t,state,pars,threshold_cfg)
        assert 0<=u<=1 and state['used']+u<=BUDGET+1e-8
        if u>0:
            assert t-state['last_on']>=REFRACTORY
            state['last_on']=t
        state['used']+=u
        rho=.89 if scenario=='ACTUATOR_SLOW' else .35
        state['z']=rho*state['z']+(1-rho)*u
        gain=true_gain(scenario,t,pars)
        old=hist[-1]
        if plant=='AR1':
            natural=nominal_forecast(old,pars)+ep[t]+sh[t]
        else:
            # Empirical replay drives the plant, not the controller.
            natural=float(replay[t])+.35*pars['phi']*(old-prev_replay)+.25*ep[t]+sh[t]
        nxt=float(np.clip(natural+gain*state['z'],0,25.))
        prev_replay=float(replay[t])
        if policy=='MPC_RISK_ADAPTIVE':
            bayes_update(state,pars,old,nxt,state['z'])
        hist.append(nxt)
        values.append(nxt);actions.append(u)
        abstentions.append(int(abstain))
        forbidden_actions.append(int((gain>0)&(u>0)))
        time_bad.append(int(gain>0))
        if with_trace:
            trace.append({'fold':fold,'test_session':FOLDS[fold]['test_session'],'plant':plant,'scenario':scenario,
                          'policy':policy,'anchor_id':anchor['anchor_id'],'seed':seed,'step':t,'t_sec':t,
                          'simulated_rate_Hz':nxt,'dimensionless_action':u,'ASSUMED_true_gain_Hz_per_unit':gain,
                          'estimated_gain_Hz_per_unit':state['mu_g'] if policy.startswith('MPC_RISK_') else np.nan})
        state['last_action']=u
    y=np.array(values,float);u=np.array(actions,float)
    target=pars['target'];scale=pars['scale']
    # Squared-excess is the primary efficacy target, not generic seizure suppression.
    excess=np.maximum(y-target,0)
    result={'fold':fold,'train_session':FOLDS[fold]['train_session'],'test_session':FOLDS[fold]['test_session'],
            'plant':plant,'scenario':scenario,'policy':policy,'anchor_id':anchor['anchor_id'],'seed':seed,
            'starting_t_sec':anchor['start_t_sec'],'mean_simulated_rate_Hz':float(y.mean()),
            'mean_excess_Hz':float(excess.mean()),'mean_squared_normalized_excess':float(np.mean((excess/scale)**2)),
            'fraction_above_target':float(np.mean(y>target)),
            'max_simulated_rate_Hz':float(y.max()),
            'total_dimensionless_action':float(u.sum()),
            'action_fraction':float(np.mean(u>0)),
            'actions_during_true_excitation':int(sum(forbidden_actions)),
            'post_reversal_action':float(np.sum(u[STEPS//3:])) if scenario=='SIGN_REVERSAL' else np.nan,
            'abstain_fraction':float(np.mean(abstentions)),
            'finite_state':bool(np.isfinite(y).all() and (y>=0).all()),
            'budget_ok':bool(u.sum()<=BUDGET+1e-8)}
    return result,trace

## DEVELOPMENT-ONLY tuning: fair threshold and dose-matched comparator

In [7]:
def tune_development(fold,plant,pars,devanchors):
    # All hyperparameters and threshold choices use training-session anchors only.
    x=pd.concat([observed[observed.fold==3-fold]],ignore_index=True)
    ths=np.unique(np.quantile(x.observed_Hz,[.35,.45,.55,.65,.75,.85,.93]))
    configs=[{'threshold':float(th),'amplitude':amp} for th in ths for amp in (.5,1.)]
    # Nominal MPC is predefined; calculate its development dose for comparator matching.
    stub={'tuned':configs[0],'dose_matched':configs[0]}
    nominal=[]
    for anchor in devanchors:
        for seed in range(N_DEV_SEEDS):
            rr,_=simulate_episode(anchor,fold,seed,plant,'STRONG_INHIBITION','MPC_NOMINAL',pars,stub)
            nominal.append(rr)
    nominal=pd.DataFrame(nominal)
    target_dose=float(nominal.total_dimensionless_action.mean())
    results=[]
    for config in configs:
        stub={'tuned':config,'dose_matched':config}
        vals=[]
        for a in devanchors:
            for seed in range(N_DEV_SEEDS):
                rr,_=simulate_episode(a,fold,seed,plant,'STRONG_INHIBITION','THRESHOLD_TUNED',pars,stub)
                vals.append(rr)
        m=pd.DataFrame(vals)
        J=float(m.mean_squared_normalized_excess.mean()+LAMBDA_ACTION*m.total_dimensionless_action.mean()/STEPS)
        results.append((config,J,float(m.total_dimensionless_action.mean()),float(m.mean_excess_Hz.mean())))
    best=min(results,key=lambda v:(v[1],v[2]))
    # Dose matching is selected independently from the performance-optimized threshold.
    dose=min(results,key=lambda v:(abs(v[2]-target_dose),v[1]))
    return ({'tuned':best[0],'dose_matched':dose[0]},
           {'fold':fold,'plant':plant,'development_session':FOLDS[fold]['train_session'],
            'threshold_tuned_J':best[1],'dose_match_target_MPC':target_dose,
            'dose_match_achieved_threshold':dose[2],
            'dose_match_abs_error':abs(dose[2]-target_dose),
            'dose_match_rel_error':abs(dose[2]-target_dose)/max(target_dose,.1),
            'tuned_threshold':best[0]['threshold'],'tuned_amplitude':best[0]['amplitude'],
            'matched_threshold':dose[0]['threshold'],'matched_amplitude':dose[0]['amplitude'],
            'n_threshold_grid_points':len(configs)})

## Run the pre-specified benchmark on held-out session anchors

In [8]:
threshold_settings={};dev_audit=[]
for fold,info in FOLDS.items():
    for plant in PLANTS:
        setting,audit=tune_development(fold,plant,info['pars'],info['dev'])
        threshold_settings[(fold,plant)]=setting;dev_audit.append(audit)
        print('Development fold',fold,plant,'MPC dose',round(audit['dose_match_target_MPC'],3),
              'matched dose',round(audit['dose_match_achieved_threshold'],3),
              'relative gap',round(audit['dose_match_rel_error'],3))
dev=pd.DataFrame(dev_audit);dev.to_csv(OUT/'T0_DEVELOPMENT_ONLY_Threshold_Dose_Matching.csv',index=False)

ROWS=[];TRACES=[]
for fold,info in FOLDS.items():
    for plant in PLANTS:
        for scenario in SCENARIOS:
            for anchor in info['test']:
                for seed in range(N_SEEDS):
                    for policy in POLICIES:
                        keep_trace=(anchor['anchor_id']==0 and seed==0 and scenario in ('STRONG_INHIBITION','SIGN_REVERSAL','EXCITATION'))
                        r,tr=simulate_episode(anchor,fold,seed,plant,scenario,policy,info['pars'],threshold_settings[(fold,plant)],keep_trace)
                        ROWS.append(r);TRACES.extend(tr)
    print('Completed held-out fold',fold)
raw=pd.DataFrame(ROWS)
traces=pd.DataFrame(TRACES)
keys=['fold','plant','scenario','anchor_id','seed']
assert raw.groupby(keys).policy.nunique().eq(len(POLICIES)).all()
assert raw.finite_state.all() and raw.budget_ok.all()
base=raw.query('policy=="NO_LIGHT"')[keys+['mean_simulated_rate_Hz','mean_excess_Hz','mean_squared_normalized_excess']].rename(columns={
    'mean_simulated_rate_Hz':'baseline_mean_rate_Hz',
    'mean_excess_Hz':'baseline_mean_excess_Hz',
    'mean_squared_normalized_excess':'baseline_mean_sq_excess'})
raw=raw.merge(base,on=keys,validate='many_to_one')
raw['delta_mean_rate_Hz_vs_no_light']=raw.mean_simulated_rate_Hz-raw.baseline_mean_rate_Hz
raw['suppression_excess_Hz_vs_no_light']=raw.baseline_mean_excess_Hz-raw.mean_excess_Hz
raw['harmful_vs_no_light']=raw.delta_mean_rate_Hz_vs_no_light>(.05*raw.fold.map({f:v['pars']['scale'] for f,v in FOLDS.items()}))
raw['objective_J']=raw.mean_squared_normalized_excess+LAMBDA_ACTION*raw.total_dimensionless_action/STEPS
z=raw[raw.scenario=='ZERO_EFFECT']
max_err=z.groupby(['fold','plant','anchor_id','seed']).mean_simulated_rate_Hz.agg(lambda s:s.max()-s.min()).max()
assert max_err<1e-10, f'Zero-effect invariance failed: {max_err}'
assert raw.query('policy=="NO_LIGHT"').total_dimensionless_action.eq(0).all()
raw.to_csv(OUT/'T1_Heldout_Paired_Simulation_Episodes.csv',index=False)
traces.to_csv(OUT/'T2_Simulation_Only_Trajectories.csv',index=False)
print('Held-out episodes',len(raw),'zero-effect max error',max_err)

Development fold 1 AR1 MPC dose 7.75 matched dose 7.6 relative gap 0.019
Development fold 1 REPLAY MPC dose 8.3 matched dose 7.85 relative gap 0.054
Development fold 2 AR1 MPC dose 4.675 matched dose 5.1 relative gap 0.091
Development fold 2 REPLAY MPC dose 4.9 matched dose 5.0 relative gap 0.02
Completed held-out fold 1
Completed held-out fold 2
Held-out episodes 16128 zero-effect max error 0.0


## Descriptive paired bootstrap by recorded starting segment (not independent animals)

In [9]:
def cluster_boot_mean(dat,val,reps,seed):
    """Vectorized two-level descriptive bootstrap: time anchors then nested seeds."""
    rng=np.random.default_rng(seed)
    groups=[g.sort_values('seed')[val].to_numpy(float) for _,g in dat.groupby('anchor_id')]
    if not groups:return np.nan,np.nan
    n=len(groups)
    ns=len(groups[0]);assert all(len(v)==ns for v in groups)
    a=np.stack(groups)
    anchor_idx=rng.integers(0,n,size=(reps,n))
    seed_idx=rng.integers(0,ns,size=(reps,n,ns))
    samples=a[anchor_idx[:,:,None],seed_idx]
    boot=samples.mean(axis=(1,2))
    return float(np.quantile(boot,.025)),float(np.quantile(boot,.975))

summary=[]
for (fold,plant,scenario,policy),g in raw.groupby(['fold','plant','scenario','policy'],sort=True):
    lo,hi=cluster_boot_mean(g,'delta_mean_rate_Hz_vs_no_light',BOOT_REPS,SEED+fold+len(plant)+len(scenario)+len(policy))
    summary.append({'fold':fold,'test_session':g.test_session.iloc[0],
                    'plant':plant,'scenario':scenario,'policy':policy,
                    'n_simulated_episodes':len(g),'n_recorded_starting_segments':g.anchor_id.nunique(),
                    'n_noise_seeds_per_segment':g.seed.nunique(),
                    'mean_rate_Hz':g.mean_simulated_rate_Hz.mean(),
                    'mean_suppression_excess_Hz':g.suppression_excess_Hz_vs_no_light.mean(),
                    'mean_action':g.total_dimensionless_action.mean(),
                    'harm_episode_fraction':g.harmful_vs_no_light.mean(),
                    'mean_delta_rate_vs_no_light_Hz':g.delta_mean_rate_Hz_vs_no_light.mean(),
                    'descriptive_95CI_delta_rate_low_Hz':lo,
                    'descriptive_95CI_delta_rate_high_Hz':hi,
                    'mean_synthetic_cost_J':g.objective_J.mean(),
                    'mean_actions_during_excitation':g.actions_during_true_excitation.mean(),
                    'mean_action_after_sign_reversal':g.post_reversal_action.mean() if scenario=='SIGN_REVERSAL' else np.nan})
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'T3_Scenario_Policy_Descriptive_Summary.csv',index=False)

# Compare risk-adaptive directly against nominal MPC and development-dose-matched threshold, paired on exact same anchor/seed.
comparisons=[]
for (fold,plant,scenario),g in raw.groupby(['fold','plant','scenario'],sort=True):
    pivot=g.pivot(index=['anchor_id','seed'],columns='policy',values=['mean_simulated_rate_Hz','mean_excess_Hz','total_dimensionless_action','harmful_vs_no_light','objective_J']).sort_index()
    for competitor in ('MPC_NOMINAL','THRESHOLD_DOSE_MATCHED','THRESHOLD_TUNED','MPC_RISK_STATIC'):
        diff=pd.DataFrame(index=pivot.index)
        for col in ('mean_simulated_rate_Hz','mean_excess_Hz','total_dimensionless_action','harmful_vs_no_light','objective_J'):
            diff[col]=pivot[(col,'MPC_RISK_ADAPTIVE')].to_numpy(float)-pivot[(col,competitor)].to_numpy(float)
        diff=diff.reset_index()
        lo,hi=cluster_boot_mean(diff,'mean_simulated_rate_Hz',BOOT_REPS,SEED+100*fold+len(plant)+len(scenario)+len(competitor))
        h_lo,h_hi=cluster_boot_mean(diff,'harmful_vs_no_light',BOOT_REPS,SEED+200*fold+len(plant)+len(scenario)+len(competitor))
        comparisons.append({'fold':fold,'plant':plant,'scenario':scenario,'comparison':'RISK_ADAPTIVE minus '+competitor,
                            'paired_mean_delta_rate_Hz':diff.mean_simulated_rate_Hz.mean(),
                            'paired_rate_diff_CI95_low_Hz':lo,'paired_rate_diff_CI95_high_Hz':hi,
                            'paired_harm_fraction_difference':diff.harmful_vs_no_light.mean(),
                            'paired_harm_diff_CI95_low':h_lo,'paired_harm_diff_CI95_high':h_hi,
                            'paired_action_difference':diff.total_dimensionless_action.mean(),
                            'paired_objective_J_difference':diff.objective_J.mean(),
                            'n_time_anchors':diff.anchor_id.nunique(),
                            'n_simulated_noise_seeds':diff.seed.nunique()})
comparisons=pd.DataFrame(comparisons)
comparisons.to_csv(OUT/'T4_Paired_MPC_vs_Threshold_and_Risk_Comparisons.csv',index=False)

## Pre-registered evidence gates with automatic PASS / FAIL / INCONCLUSIVE labels

In [10]:
# This is a computational hypothesis test, NOT clinical validation.
GATES=[]
for plant in PLANTS:
    for fold in (1,2):
        s=summary.query('plant==@plant and fold==@fold')
        def get(scen,pol,metric):
            return float(s[(s.scenario==scen)&(s.policy==pol)].iloc[0][metric])
        inh_nom=get('STRONG_INHIBITION','MPC_NOMINAL','mean_suppression_excess_Hz')
        inh_risk=get('STRONG_INHIBITION','MPC_RISK_ADAPTIVE','mean_suppression_excess_Hz')
        exc_harm_nom=get('EXCITATION','MPC_NOMINAL','harm_episode_fraction')
        exc_harm_risk=get('EXCITATION','MPC_RISK_ADAPTIVE','harm_episode_fraction')
        rev_harm_nom=get('SIGN_REVERSAL','MPC_NOMINAL','harm_episode_fraction')
        rev_harm_risk=get('SIGN_REVERSAL','MPC_RISK_ADAPTIVE','harm_episode_fraction')
        thr_dose=get('STRONG_INHIBITION','THRESHOLD_DOSE_MATCHED','mean_action')
        mpc_dose=get('STRONG_INHIBITION','MPC_NOMINAL','mean_action')
        ratio=(inh_risk/inh_nom) if inh_nom>0.01 else np.nan
        # Engineering diagnostic thresholds chosen BEFORE viewing held-out test results.
        criteria={
           'Retains >=80% of nominal MPC strong-inhibition suppression':bool(np.isfinite(ratio) and ratio>=.8),
           'Lower harmful-episode fraction under excitation':bool(exc_harm_risk<exc_harm_nom-1e-8),
           'Lower harmful-episode fraction under sign reversal':bool(rev_harm_risk<rev_harm_nom-1e-8),
           'Threshold development-dose-match within 20% on test':bool(abs(thr_dose-mpc_dose)/max(mpc_dose,.1)<=.20),
           'Zero-effect trajectory invariance':bool(max_err<1e-10),
        }
        for k,passed in criteria.items():
            GATES.append({'fold':fold,'plant':plant,'criterion':k,'status':'PASS' if passed else 'FAIL',
                          'note':'Descriptive simulated criterion only; not animal- or patient-level inference'})
gates=pd.DataFrame(GATES)
gates.to_csv(OUT/'T5_Predeclared_Engineering_Evidence_Gates.csv',index=False)

## Paper figures (simulation results only)

In [11]:
plt.rcParams.update({'font.size':9,'axes.titlesize':11,'savefig.dpi':220})
colors={'NO_LIGHT':'#838b99','THRESHOLD_TUNED':'#6681c2','THRESHOLD_DOSE_MATCHED':'#4b9c95','MPC_NOMINAL':'#d17a51','MPC_RISK_STATIC':'#9974b8','MPC_RISK_ADAPTIVE':'#2c6e96','FIXED':'#b5a475'}

# Fig1: experimental design diagram, simple and explicit.
fig,ax=plt.subplots(figsize=(10,4));ax.axis('off')
blocks=[('Phase 5A frozen\nobserved firing rates',.08),('Training-session\npolicy tuning',.32),('Held-out-session\npaired scenarios',.56),('Policy ranking\n+ failure cases',.80)]
for s,x in blocks:
    ax.text(x,.55,s,va='center',ha='center',fontsize=10,weight='bold',
            bbox={'boxstyle':'round,pad=.6','facecolor':'#eef2f6','edgecolor':'#93a5b8'})
for x in (.19,.43,.67):ax.annotate('',xy=(x+.06,.55),xytext=(x-.015,.55),arrowprops={'arrowstyle':'->','lw':2,'color':'#60778a'})
ax.text(.5,.08,'Artificial intervention gains; one mouse; no causal or clinical efficacy claims',ha='center',fontsize=10,color='#9d3842')
fig.tight_layout();fig.savefig(OUT/'Fig1_Benchmark_Design_SIMULATION.png');fig.savefig(OUT/'Fig1_Benchmark_Design_SIMULATION.svg');plt.close(fig)

# Fig2 harm fraction under incorrect-sign scenario; separate plants, folds.
fig,axs=plt.subplots(2,2,figsize=(11,7),sharey=True)
for i,fold in enumerate((1,2)):
    for j,plant in enumerate(PLANTS):
        ax=axs[i,j];g=summary[(summary.fold==fold)&(summary.plant==plant)&summary.scenario.isin(['EXCITATION','SIGN_REVERSAL'])]
        pol=['THRESHOLD_DOSE_MATCHED','MPC_NOMINAL','MPC_RISK_STATIC','MPC_RISK_ADAPTIVE']
        xx=np.arange(2);w=.19
        for k,p in enumerate(pol):
            gg=g[g.policy==p].set_index('scenario').loc[['EXCITATION','SIGN_REVERSAL']]
            ax.bar(xx+(k-1.5)*w,gg.harm_episode_fraction,width=w,color=colors[p],label=p.replace('_',' '))
        ax.set_xticks(xx,['Excitatory gain','Sign reversal']);ax.set_ylim(0,1.08)
        ax.set_title(f'Fold {fold}, {plant} plant')
        ax.grid(axis='y',alpha=.17)
axs[0,0].set_ylabel('Simulated harmful-episode fraction');axs[1,0].set_ylabel('Simulated harmful-episode fraction')
fig.legend(*axs[0,0].get_legend_handles_labels(),loc='lower center',ncol=4,bbox_to_anchor=(.5,-.01),fontsize=8)
fig.suptitle('SIMULATION ONLY | Harm = mean activity > no-light by 0.05 x training-rate SD')
fig.tight_layout(rect=[0,.05,1,.95]);fig.savefig(OUT/'Fig2_Sign_Mismatch_Harm_SIMULATION.png',bbox_inches='tight',pad_inches=.18);fig.savefig(OUT/'Fig2_Sign_Mismatch_Harm_SIMULATION.svg',bbox_inches='tight',pad_inches=.18);plt.close(fig)

# Fig3 dose vs excess (proper trade-off; threshold dose match highlighted)
fig,axs=plt.subplots(2,2,figsize=(10,7))
for i,fold in enumerate((1,2)):
    for j,plant in enumerate(PLANTS):
        ax=axs[i,j];g=summary.query('fold==@fold and plant==@plant and scenario=="STRONG_INHIBITION"')
        for r in g.itertuples():
            ax.scatter(r.mean_action,r.mean_suppression_excess_Hz,s=50,color=colors[r.policy],label=r.policy.replace('_',' '))
        ax.set_title(f'Fold {fold}, {plant}')
        ax.axhline(0,lw=1,color='#aaaaaa');ax.grid(alpha=.2)
fig.supxlabel('Mean total dimensionless stimulation per episode', fontsize=10, y=.055)
fig.supylabel('Reduction in simulated excess above target (Hz)', fontsize=10, x=.004)
handles,labels=axs[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=4,fontsize=8,bbox_to_anchor=(.5,-.02))
fig.suptitle('SIMULATION ONLY | Dose–efficacy trade-off under assumed inhibition')
fig.tight_layout(rect=[0,.06,1,.95]);fig.savefig(OUT/'Fig3_Dose_Efficacy_Pareto_SIMULATION.png',bbox_inches='tight',pad_inches=.18);fig.savefig(OUT/'Fig3_Dose_Efficacy_Pareto_SIMULATION.svg',bbox_inches='tight',pad_inches=.18);plt.close(fig)

# Fig4 signed sensitivity across two scenarios, per fold for REPLAY plant.
fig,axs=plt.subplots(1,2,figsize=(11,4.3),sharey=True)
for idx,fold in enumerate((1,2)):
    ax=axs[idx]
    g=summary.query('plant=="REPLAY" and fold==@fold')
    order=['STRONG_INHIBITION','WEAK_INHIBITION','ZERO_EFFECT','EXCITATION','SIGN_REVERSAL','ACTUATOR_SLOW']
    xx=np.arange(len(order))
    for p in ('THRESHOLD_DOSE_MATCHED','MPC_NOMINAL','MPC_RISK_ADAPTIVE'):
        gp=g[g.policy==p].set_index('scenario').loc[order]
        ax.plot(xx,gp.mean_delta_rate_vs_no_light_Hz,marker='o',label=p.replace('_',' '),color=colors[p])
    ax.set_xticks(xx,['strong\ninh','weak\ninh','zero','excite','sign\nflip','slow'],fontsize=8)
    ax.axhline(0,color='black',lw=.9);ax.set_title(f'Fold {fold} (REPLAY plant)');ax.grid(alpha=.2)
axs[0].set_ylabel('Mean simulated rate minus no light (Hz)')
fig.legend(*axs[0].get_legend_handles_labels(),loc='lower center',ncol=3,fontsize=8,bbox_to_anchor=(.5,-.02))
fig.suptitle('SIMULATION ONLY | Negative values = lower activity; positive = increase')
fig.tight_layout(rect=[0,.08,1,.92]);fig.savefig(OUT/'Fig4_Controller_Gain_Stress_SIMULATION.png',bbox_inches='tight',pad_inches=.18);fig.savefig(OUT/'Fig4_Controller_Gain_Stress_SIMULATION.svg',bbox_inches='tight',pad_inches=.18);plt.close(fig)

# Fig5: representative trajectories for one held-out animal recording snippet, sign-flip.
fig,axs=plt.subplots(2,1,figsize=(10,6),sharex=True)
g=traces.query('fold==1 and plant=="REPLAY" and scenario=="SIGN_REVERSAL" and policy in ["NO_LIGHT","THRESHOLD_DOSE_MATCHED","MPC_NOMINAL","MPC_RISK_ADAPTIVE"]')
for p,grp in g.groupby('policy'):
    grp=grp.sort_values('step')
    axs[0].plot(grp.step,grp.simulated_rate_Hz,color=colors[p],label=p.replace('_',' '),lw=1.6)
    axs[1].step(grp.step,grp.dimensionless_action,where='post',color=colors[p],label=p.replace('_',' '))
for ax in axs: ax.axvline(STEPS/3,color='#c64747',linestyle='--',label='gain reverses')
axs[0].set_ylabel('Simulated firing rate (Hz)');axs[1].set_ylabel('Dimensionless action')
axs[1].set_xlabel('Simulation time (seconds)')
axs[0].legend(fontsize=8,ncol=2);axs[0].grid(alpha=.2)
fig.suptitle('SIMULATION ONLY | One illustrative paired episode, sign reversal')
fig.tight_layout();fig.savefig(OUT/'Fig5_Sign_Reversal_Trajectories_SIMULATION.png');fig.savefig(OUT/'Fig5_Sign_Reversal_Trajectories_SIMULATION.svg');plt.close(fig)

## Locked protocol, limitations, reproducibility, final archive

In [12]:
report=[]
report.append('# Phase 6A | Conference-focused controlled simulation benchmark\n')
report.append('**Question:** When do nominal and uncertainty-aware MPC outperform or fail relative to an independently tuned, dose-matched threshold policy when intervention gain or sign is wrong?\n')
report.append('## Evidence boundary\n')
report.append('The neural rate samples are real spontaneous mouse recordings (DANDI 000568, Phase 5A held-out predictions); stimulation actions, gain, actuator, dynamics following actions and all effects are entirely SIMULATED. One mouse, two sessions. Numerical seeds and segments are not independent biological replicates. No neural seizure labels or clinical claims. The controller uses a simple training-session-fitted autoregressive predictor, NOT the frozen GRU checkpoint.\n')
report.append('## Predeclared controller comparisons\n')
report.append('No light; fixed light; threshold tuned for the same development objective; threshold matched in development dose to nominal MPC; nominal short-horizon MPC; static uncertainty MPC; adaptive uncertainty MPC using an imperfect gain posterior. All receive the same causal feedback, action set, stimulation budget and refractory period. Controller hyperparameters and threshold selection use training-session snapshots only. Held-out sessions are used only for final scenarios.\n')
report.append('## Scenarios and metrics\n')
report.append('Plants: fitted AR(1) and observed-mouse-rate-anchored replay. Gains: strong inhibition, weak inhibition, no effect, excitation, sign reversal, slow actuator. Primary outcomes are simulated excess above a training-defined activity target and total dimensionless stimulation. Mismatch outcomes: harmful-episode fraction vs matched no-light trajectory, post-reversal action, intervention under excitatory sign, and paired controller differences. Dose-matching quality and zero-effect invariance are mandatory checks. Synthetic confidence intervals cluster by starting time segments and remain descriptive.\n')
report.append('## Predeclared engineering success criteria\n')
report.append('Under assumed strong inhibition, the adaptive controller should retain at least 80% of nominal MPC excess reduction; under excitation and sign reversal, its harmful-episode fraction should be lower; matched threshold and MPC should be within 20% of each other in mean action on test; zero-effect trajectories must coincide. Failing a criterion must be reported and not retroactively retuned.\n')
report.append('## Why this is not biological control\n')
report.append('No light-response model was identified from experimental interventions. Even using real spontaneous rates as a replay baseline does not validate how light would change those rates. Optical action is dimensionless and does not represent calibrated watts or safe dose. Held-out test sessions were examined in earlier project phases, so they are not pristine unseen prospective tests. Model mismatch and Bayesian posterior safety are only explored in an artificial simulator.\n')
report.append('## Prior relevant literature (do not claim this benchmark is first)\n')
report.append('- Acharya et al. (2024), evoked intracranial EEG response prediction in epilepsy: https://doi.org/10.1038/s42003-024-06859-2\n')
report.append('- Brogin et al. (2026), simulated seizure reconstruction/attenuation and control: https://www.nature.com/articles/s41598-026-39645-7\n')
report.append('- Learning temporal basis vectors for neural stimulation: https://pubmed.ncbi.nlm.nih.gov/41336632/\n')
report.append('## Numeric verification\n')
report.append(f'- Paired simulation episodes: {len(raw)}; {N_TEST_ANCHORS} recorded start histories per held-out session; {N_SEEDS} noise seeds per history.\n')
report.append(f'- Zero-effect maximum policy-difference in mean trajectory: {max_err:.3e} Hz.\n')
report.append('- All actions obey the same per-episode budget and refractory constraints.\n')
report.append('## Interpretation rule\n')
report.append('A computational advantage requires both a fair efficacy/dose comparison and a better mismatch safety trade-off, separately in each plant and fold. Do not claim a clinical safety guarantee, generalization across animals, or causal optogenetic effects.\n')
(OUT/'RESEARCH_PROTOCOL_AND_LIMITATIONS.md').write_text('\n'.join(report),encoding='utf8')
metadata={'experiment_id':'Phase6A','source_sha256':hashlib.sha256(BUNDLE.read_bytes()).hexdigest(),
          'phase5a_archive_sha256':provenance['source_archive_sha256'],
          'dataset':'DANDI 000568','species':'mouse','n_biological_animals':1,
          'controller_core':'fitted causal AR1 surrogate, not a neural-network-in-the-loop controller',
          'simulated_scenarios':SCENARIOS,'plants':PLANTS,'policies':POLICIES,
          'quick_mode':QUICK_MODE,'steps':STEPS,'seed':SEED,
          'n_episodes':len(raw),'n_time_anchors_per_fold':N_TEST_ANCHORS,'n_noise_seeds':N_SEEDS,
          'zero_effect_invariance_max_error':float(max_err),
          'published_data_code':'https://dandiarchive.org/dandiset/000568',
          'note':'real neural baseline; unverified hypothetical intervention effects'}
(OUT/'PROVENANCE_SIMULATION_ONLY.json').write_text(json.dumps(metadata,indent=2),encoding='utf8')
zipfile.ZipFile
outzip=OUT.parent/'phase6a_conference_robustness_results.zip'
with zipfile.ZipFile(outzip,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('FINISHED',outzip,'bytes',outzip.stat().st_size)
print('\nPRIMARY SIMULATED RESULTS (fold1 AR1)')
print(summary.query('fold==1 and plant=="AR1" and scenario in ["STRONG_INHIBITION","EXCITATION","SIGN_REVERSAL"] and policy in ["THRESHOLD_DOSE_MATCHED","MPC_NOMINAL","MPC_RISK_ADAPTIVE"]')
      [['scenario','policy','mean_rate_Hz','mean_action','harm_episode_fraction','mean_suppression_excess_Hz']].round(3).to_string(index=False))
print('\nGATE STATUS',gates.status.value_counts().to_dict())

FINISHED /kaggle/working/phase6a_conference_robustness_results.zip bytes 1710289

PRIMARY SIMULATED RESULTS (fold1 AR1)
         scenario                 policy  mean_rate_Hz  mean_action  harm_episode_fraction  mean_suppression_excess_Hz
       EXCITATION            MPC_NOMINAL         2.395        9.396                  1.000                      -0.142
       EXCITATION      MPC_RISK_ADAPTIVE         2.411       10.120                  1.000                      -0.156
       EXCITATION THRESHOLD_DOSE_MATCHED         2.428       11.115                  1.000                      -0.169
    SIGN_REVERSAL            MPC_NOMINAL         2.278        9.052                  0.490                      -0.069
    SIGN_REVERSAL      MPC_RISK_ADAPTIVE         2.279       11.188                  0.469                      -0.082
    SIGN_REVERSAL THRESHOLD_DOSE_MATCHED         2.320       10.385                  0.760                      -0.101
STRONG_INHIBITION            MPC_NOMINAL       

## How to use the outputs

In Kaggle, leave `QUICK_MODE = False`, select **CPU / Accelerator None**, then click **Run All**. After completion, download `/kaggle/working/phase6a_conference_robustness_results.zip`. All tables and figures are from simulated interventions. **DO NOT** interpret simulated harms as biological adverse events, predicted activity reductions as epilepsy treatment, or numerical confidence intervals as independent-mouse inference.

Final benchmark: 2 session-direction folds × 2 synthetic plant types × 6 intervention scenarios × 7 controllers × 16 observed starting segments × 6 noise seeds. Policies share disturbance seeds and experimental action constraints. Threshold selection is restricted to the other recording session.

**Do not upload previously generated local or synthetic-output ZIPs as the Phase 6A experimental result.**
